# 11 · Tuned RET and untuned pairwise ranking baseline

Two new experiments on the immutable 42-feature panel. Each has 100 members per quarter, 40 quarters, 12-month training/3-month forward blocks, and N=2/3 equal-weight long/short portfolios. The existing completed RET42 run remains a historical reference.

RET tuning evaluates all 24 declared settings on three historical whole-day splits per quarter (2,880 exploratory fits). It selects by validation log loss, then fits the full 100-member ensemble with the original row-split schedule. The ranker has fixed untuned parameters, whole-day queries and symmetric top/bottom-three validation NDCG. The direct learner comparison also changes validation protocol; it is not an isolated objective-only experiment.

All features, forecasts, outcomes and trading assumptions remain unchanged. No network or one-minute work. No round, RAM, memory-headroom or wall-clock ceilings. This notebook only inspects results; it never fits or rebuilds missing inputs.

In [ ]:
from pathlib import Path
import sys,json,os
PROJECT_DIR=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"scripts/learner_experiments.py").exists())
SUITE=PROJECT_DIR/"data/models/11_ret_ranker_parallel_v1"
REPORT_DIR=PROJECT_DIR/"data/reports/11_ret_tuning_and_ranker"
REPORT_DIR.mkdir(parents=True,exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR",str(REPORT_DIR/".matplotlib"))
FROZEN=SUITE/"source_snapshot/scripts"
sys.path.insert(0,str(FROZEN if FROZEN.exists() else PROJECT_DIR/"scripts"))
import pandas as pd
from IPython.display import display,Image,Markdown
RUN_TRAINING=False
COMPLETE=all((SUITE/v/"RESULTS.json").exists() and json.loads((SUITE/v/"RUN_STATUS.json").read_text())["status"]=="complete" for v in ["ret_tuned","ranker"])
if (SUITE/"RUN_STATUS.json").exists():
    display(pd.DataFrame.from_dict(json.loads((SUITE/"RUN_STATUS.json").read_text()),orient="index",columns=["suite_status"]))
if not COMPLETE:display(Markdown("Final results are withheld until both 4,000-member runs complete. Inspection does not start training."))

## Conditional matched-date portfolios

Compare all three models × N=2/3 on the same observed dates. Retain cash dates; explicitly record omitted dates and selected unknown exits. These curves do not establish a full executable path. Entry/exit dollar-notional costs are sensitivities, without actual spread, slippage or borrow.

The current gap observes the opening field and assumes a same-price fill; the published bar arrives later. This remains optimistic research.

In [ ]:
if COMPLETE:
    from learner_experiments import report_suite
    output=report_suite(PROJECT_DIR)
    display(pd.read_parquet(output/"common_date_comparison.parquet"))
    display(Image(filename=str(output/"portfolio_comparison.png")))
    display(pd.read_csv(output/"selected_hyperparameters.csv"))
    display(pd.DataFrame({name:json.loads((folder/"RESULTS.json").read_text())["classification"] for name,folder in [("historical_RET42",PROJECT_DIR/"data/models/10_xgb_ret_rich_path100_v1"),("tuned_RET42",SUITE/"ret_tuned"),("ranker42",SUITE/"ranker")]}))

## Feature use and forecast fit

Importance normalizes training split gain within each of 4,000 saved models. The classifier and ranker use different objectives; gain shares are not causal effects or explained return variance.

Return EV/R² are reported only for historical and tuned RET. Ranker outputs are arbitrary ranking margins, converted into within-day member percentiles for averaging. They are not expected returns; no uncalibrated EV/R² or fake class probabilities are published. Ranker classification accuracy/log loss are undefined. Its observed-day Spearman IC and selected portfolio returns remain available.

RET member split/seed and forecast-row fingerprints and quarterly payoff vectors must match the completed RET42 reference. Ranker splits intentionally keep whole dates together. Tuning validation scores were used for selection and are not unbiased holdout results. Previously viewed 2017–2026 history remains development evidence.

In [ ]:
if COMPLETE:
    display(pd.read_parquet(output/"return_fit/explained_variance.parquet"))
    for name in ["feature_importance.png","ret_explained_variance.png"]:display(Image(filename=str(output/name)))
    display(pd.DataFrame.from_dict(json.loads((output/"COMPARISON_AUDIT.json").read_text()),orient="index",columns=["audit"]))